# DEAD proposal stuff

In [ ]:
import duckdb as db
import pandas as pd
import numpy as np
con = db.connect()

## Download + Parquet

In [ ]:
import duckdb

con = duckdb.connect()

# NOTE: this direct-from-URL approach failed with a CSV "sniffing" error.
# That error almost always means the URL is NOT returning a clean CSV --
# e.g. the Iowa data portal is returning an HTML error/redirect page,
# a JSON payload, or a gzip/paginated response instead of plain CSV.
# Before re-trying this, open one of the URLs in a browser (or curl -I it)
# and confirm you actually get a CSV file back, not a webpage.
#
# If you DO get a real CSV and it still fails to parse, try being explicit
# instead of relying on auto-detection, e.g.:
#
# con.execute("""
#     COPY (
#         SELECT *, 2022 AS source_year
#         FROM read_csv(
#             'https://idh-be.iowa.gov/api/v1/datasets/1259/rows.csv',
#             delim=',',
#             quote='\"',
#             ignore_errors=true,
#             max_line_size=10000000
#         )
#     )
#     TO 'liquor_2022.parquet'
#     (FORMAT PARQUET);
# """)
#
# For now we're using the manually-exported JSON approach below instead,
# which sidesteps this URL-parsing issue entirely.


What factors drive higher or lower alcohol sales?

In [ ]:
import zipfile
import glob
import os
import duckdb

# --- 1) Locate the zip file ---
# The FileNotFoundError means Python looked for the zip in your CURRENT
# WORKING DIRECTORY (wherever you launched Jupyter from) and didn't find it
# there. It does NOT mean the file doesn't exist anywhere on your machine --
# it's very likely still sitting in your Downloads folder (or wherever your
# OneDrive sync saves it), just not next to this notebook.

zip_name = "OneDrive_1_9-10-2026.zip"

# Places to check, in order. Add/edit paths here if yours lives somewhere else.
candidate_dirs = [
    ".",                                   # same folder as this notebook
    os.path.expanduser("~/Downloads"),
    os.path.expanduser("~/OneDrive"),
    os.path.expanduser("~/OneDrive/Downloads"),
]

zip_path = None
for d in candidate_dirs:
    candidate = os.path.join(d, zip_name)
    if os.path.isfile(candidate):
        zip_path = candidate
        break

if zip_path is None:
    # Last resort: search a couple levels down from the home directory
    matches = glob.glob(os.path.expanduser(f"~/**/{zip_name}"), recursive=True)
    if matches:
        zip_path = matches[0]

if zip_path is None:
    raise FileNotFoundError(
        f"Could not find '{zip_name}' automatically. "
        f"Current working directory is: {os.getcwd()}\n"
        f"Run `os.listdir('.')` to see what's actually here, find where the "
        f"zip downloaded to, and either move it next to this notebook or set "
        f"zip_path = r'/full/path/to/{zip_name}' manually below."
    )

print(f"Using zip file: {zip_path}")

extract_path = "liquor_2022_2026"

# --- 2) Unzip ---
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# --- 3) Find every JSON file inside all subfolders ---
json_files = glob.glob(f"{extract_path}/**/*.json", recursive=True)
print(f"Found {len(json_files)} JSON files")

if not json_files:
    raise FileNotFoundError(
        f"Unzipped '{zip_path}' into '{extract_path}' but found no .json files. "
        f"Check that this is the right export (JSON, not CSV/XLSX) and that "
        f"the zip actually contains data files and not just a single wrapper folder."
    )

# --- 4) Convert all JSON files into one Parquet ---
con = duckdb.connect()

con.execute(
    """
    COPY (
        SELECT *
        FROM read_json_auto(
            ?,
            union_by_name=true
        )
    )
    TO 'liquor_2022_2026.parquet'
    (FORMAT PARQUET)
    """,
    [json_files],
)

print("Done! Created liquor_2022_2026.parquet")

# --- 5) Quick sanity check: peek at the schema/columns you'll be working with ---
preview = con.execute("SELECT * FROM 'liquor_2022_2026.parquet' LIMIT 5").df()
print(preview.columns.tolist())
preview


## Inspect schema before aggregating

Column names below are my best guess based on the standard Iowa liquor sales schema (Socrata export). **Run this first and check the printed column list against the SQL below** — rename anything that doesn't match your actual file.

In [ ]:
import duckdb

con = duckdb.connect()

# Peek at the actual columns/types in your file
print(con.execute("DESCRIBE SELECT * FROM 'liquor_2022_2026.parquet'").df())
con.execute("SELECT * FROM 'liquor_2022_2026.parquet' LIMIT 3").df()


## Build the Category x Month aggregated table

Observational unit: **Alcohol Category x Year-Month**.

Note: I'm aggregating by *year-month* (e.g. `2023-06`), not just calendar month
(e.g. `June`) collapsed across years -- if you collapse across years you throw
away 2022 vs 2026 trend/growth, which is almost certainly something DEAD cares
about. If you actually want pure seasonality (one row per category per
calendar month, pooling all years together), swap `year_month` for `month_num`
in the GROUP BY -- I left `month_num` and `year` as separate columns either way
so you can slice it either direction later.

**Column names to double-check against your DESCRIBE output above:**
- `date` -> the sale date column
- `category_name` -> the liquor category text field (sometimes just `category`)
- `bottles_sold` -> unit count
- `sale_dollars` -> dollar amount
- `volume_sold_liters` -> volume
- `state_bottle_retail` -> per-bottle retail price
- `store_number`, `zip_code` -> used only to build diversity/reach features, not as grouping keys


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE agg AS
    SELECT
        category_name,
        strftime(CAST(date AS DATE), '%Y-%m')          AS year_month,
        EXTRACT(year  FROM CAST(date AS DATE))          AS year,
        EXTRACT(month FROM CAST(date AS DATE))          AS month_num,

        -- === targets ===
        SUM(bottles_sold)                               AS total_units,
        SUM(sale_dollars)                                AS total_dollars,

        -- === supporting features, aggregated to the same grain ===
        SUM(volume_sold_liters)                          AS total_liters,
        AVG(state_bottle_retail)                         AS avg_bottle_price,
        SUM(sale_dollars) / NULLIF(SUM(bottles_sold), 0) AS realized_price_per_unit,
        COUNT(DISTINCT store_number)                     AS n_stores,
        COUNT(DISTINCT zip_code)                         AS n_zips_reached,
        COUNT(*)                                         AS n_transactions

    FROM 'liquor_2022_2026.parquet'
    GROUP BY category_name, year_month, year, month_num
    ORDER BY category_name, year_month
""")

df = con.execute("SELECT * FROM agg").df()
print(df.shape)
df.head(10)


## Target variable prep

Two candidate targets per row (`total_units`, `total_dollars`). As discussed:
`total_dollars` conflates volume with price/premiumization, so I'd treat
`total_units` (or `total_liters` if you want an even cleaner consumption
measure) as primary, and `total_dollars` as a secondary/parallel target if you
also want to speak to spending.

Both are log-transformed below (`log1p` handles any zero-sale category-months
safely) since raw sales counts are typically right-skewed across categories --
this will matter once you're checking linear regression residuals.


In [ ]:
import numpy as np

df["log_units"]   = np.log1p(df["total_units"])
df["log_dollars"] = np.log1p(df["total_dollars"])
df["log_liters"]  = np.log1p(df["total_liters"])

# Quick look at skew before/after log transform
print(df[["total_units", "log_units", "total_dollars", "log_dollars"]].describe())

df.to_parquet("liquor_category_month_agg.parquet", index=False)
print("Saved liquor_category_month_agg.parquet")


## Optional: seasonality / trend features for the regression

Since your obs unit is time-based (year-month per category), these are the
natural linear-regression-friendly features to add before modeling -- month
dummies to capture seasonality (holidays, summer, etc.) and a simple linear
time trend to capture year-over-year growth/decline, plus a category dummy
set since "category" itself is a huge driver of both units and price.


In [ ]:
# Linear time trend (months since the start of the panel)
df["t"] = (df["year"] - df["year"].min()) * 12 + df["month_num"]

# One-hot encode month (seasonality) and category (product mix) for use
# directly in a linear regression / OLS design matrix
model_df = pd.get_dummies(
    df,
    columns=["month_num", "category_name"],
    drop_first=True,
)

model_df.head()
